# SIGNOVA — Incremental & Resource-Aware Training Pipeline

This notebook implements the continuous sign language training pipeline for **ArabSign (ArSL)** and **How2Sign (ASL)**.

### Features:
- **Hardware Acceleration:** Native PyTorch CUDA support on local GPU (RTX 5060) or Google Colab / Kaggle GPU.
- **Strict Linguistic Separation:** Independent vocabulary and loss computation for ArabSign and How2Sign.
- **Signer-Independent Protocol:** Zero data leakage across training, validation, and test partitions.
- **Resource-Aware:** Configurable batch sizes and sample limits to prevent RAM/VRAM saturation.

In [1]:
import sys
from pathlib import Path

# Fix Windows console encoding for Arabic characters
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

import torch
import yaml

# Robust repository root resolution
def find_repo_root():
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "pyproject.toml").exists() and (p / "ml").exists():
            return p
    return cur

repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project Root: {repo_root}")
print(f"Device selected for training: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # Blackwell architecture fix for PyTorch sequence kernels
    torch.backends.cudnn.enabled = False

Project Root: C:\Users\moham\Desktop\DEPI\Microsoft Machine Learning\Technical\Final Project\SIGNOVA
Device selected for training: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## 1. Load Training Configuration

In [2]:
config_path = repo_root / "ml/sign_language/configs/config.yaml"
with open(config_path, encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("Model Configuration:", config["model"])
print("Training Hyperparameters:", config["training"])

Model Configuration: {'architecture': 'BiLSTM_CTC', 'input_dim': 126, 'hidden_dim': 256, 'num_layers': 2, 'dropout': 0.2, 'bidirectional': True}
Training Hyperparameters: {'batch_size': 16, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'max_epochs': 10, 'gradient_clip_norm': 5.0, 'nan_strategy': 'interpolate', 'checkpoint_dir': 'ml/sign_language/artifacts', 'save_best_only': True, 'metric_for_best': 'val_loss'}


## 2. Execute Training: ArabSign (Continuous Arabic Sign Language - ArSL)

In [3]:
from ml.sign_language.src.train import run_training

print("=== Starting ArabSign (ArSL) Training ===")
arab_results = run_training(
    dataset_name="arabsign",
    epochs=5,
    batch_size=16,
    max_samples=None,  # Full 6,323 training samples
)

print("\nArabSign Training Completed!")
print(f"Best Validation Loss: {arab_results['best_val_loss']:.4f}")
print(f"Saved Artifacts: {arab_results['artifact_dir']}")

=== Starting ArabSign (ArSL) Training ===
🚀 Training starting on device: cuda (dataset: arabsign)
Built vocabulary: 98 unique tokens for ArSL
Loaded 6323 train samples and 1500 val samples.
Epoch [01/05] | Train Loss: 5.3049 | Val Loss: 3.8837 | Val WER: 0.8493 | Token Acc: 15.07%
  --> Saved new best checkpoint to C:\Users\moham\Desktop\DEPI\Microsoft Machine Learning\Technical\Final Project\SIGNOVA\ml\sign_language\artifacts\arabsign\best_model.pth
Epoch [02/05] | Train Loss: 3.8813 | Val Loss: 3.8114 | Val WER: 0.8493 | Token Acc: 15.07%
  --> Saved new best checkpoint to C:\Users\moham\Desktop\DEPI\Microsoft Machine Learning\Technical\Final Project\SIGNOVA\ml\sign_language\artifacts\arabsign\best_model.pth
Epoch [03/05] | Train Loss: 3.8257 | Val Loss: 3.7861 | Val WER: 0.8493 | Token Acc: 15.07%
  --> Saved new best checkpoint to C:\Users\moham\Desktop\DEPI\Microsoft Machine Learning\Technical\Final Project\SIGNOVA\ml\sign_language\artifacts\arabsign\best_model.pth
Epoch [04/05] |

## 3. Execute Pilot Training: How2Sign (Continuous ASL)

In [4]:
print("=== Starting How2Sign (ASL) Pilot Training ===")
how_results = run_training(
    dataset_name="how2sign",
    epochs=5,
    batch_size=16,
    max_samples=2000,  # Pilot subset
)

print("\nHow2Sign Training Completed!")
print(f"Best Validation Loss: {how_results['best_val_loss']:.4f}")
print(f"Saved Artifacts: {how_results['artifact_dir']}")

=== Starting How2Sign (ASL) Pilot Training ===
🚀 Training starting on device: cuda (dataset: how2sign)
Built vocabulary: 5856 unique tokens for ASL
Loaded 2000 train samples and 500 val samples.
Epoch [01/05] | Train Loss: 9.8763 | Val Loss: 7.3088 | Val WER: 1.0000 | Token Acc: 0.00%
  --> Saved new best checkpoint to C:\Users\moham\Desktop\DEPI\Microsoft Machine Learning\Technical\Final Project\SIGNOVA\ml\sign_language\artifacts\how2sign\best_model.pth
Epoch [02/05] | Train Loss: 6.9946 | Val Loss: 7.5039 | Val WER: 1.0000 | Token Acc: 0.00%
Epoch [03/05] | Train Loss: 6.8846 | Val Loss: 7.4491 | Val WER: 0.9971 | Token Acc: 0.29%
Epoch [04/05] | Train Loss: 6.7373 | Val Loss: 7.3982 | Val WER: 0.9971 | Token Acc: 0.29%
Epoch [05/05] | Train Loss: 6.6151 | Val Loss: 7.3573 | Val WER: 0.9960 | Token Acc: 0.40%

How2Sign Training Completed!
Best Validation Loss: 7.3088
Saved Artifacts: C:\Users\moham\Desktop\DEPI\Microsoft Machine Learning\Technical\Final Project\SIGNOVA\ml\sign_langua